# Loss Functions and Output Layers

Companion notebook for Section 5 of the MLP lecture notes.

Objectives:

- match output activation and loss to the task;
- compute MSE, binary cross-entropy, and categorical cross-entropy;
- understand probabilities versus logits;
- see why logits-aware losses are numerically stable;
- identify common wrong output/loss combinations.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)


## 1. Task-to-output checklist

| Task | Output layer | Loss | Target format |
|---|---|---|---|
| Regression | Linear | MSE | real value |
| Binary classification | Sigmoid probability or one raw logit | BCE | 0/1 |
| Multi-class classification | Softmax probabilities or K raw logits | Cross-entropy | class index or one-hot |


## 2. Mean squared error for regression

The lecture uses `0.5 * (y_hat - y)^2` for a single example.


In [ ]:
def mse_half(y_hat, y):
    y_hat = np.asarray(y_hat)
    y = np.asarray(y)
    return 0.5 * np.mean((y_hat - y) ** 2)

y_reg = np.array([2.0, 0.0, -1.0, 3.0])
y_hat_good = np.array([1.9, 0.1, -1.2, 2.8])
y_hat_bad = np.array([0.0, 2.0, 1.0, -1.0])

print('Good regression predictions loss:', mse_half(y_hat_good, y_reg))
print('Bad regression predictions loss:', mse_half(y_hat_bad, y_reg))


In [ ]:
pred_grid = np.linspace(-2, 5, 300)
y_target = 2.0
loss_grid = 0.5 * (pred_grid - y_target) ** 2

fig, ax = plt.subplots()
ax.plot(pred_grid, loss_grid)
ax.axvline(y_target, color='black', linestyle='--', label='target')
ax.set_xlabel('prediction')
ax.set_ylabel('0.5 * squared error')
ax.set_title('MSE penalizes distance from a real-valued target')
ax.legend()
plt.show()


## 3. Binary cross-entropy with probabilities

Binary cross-entropy expects a probability `p = P(y=1|x)`, usually produced by a sigmoid output.


In [ ]:
def binary_cross_entropy_from_probability(p, y, eps=1e-12):
    p = np.clip(np.asarray(p), eps, 1 - eps)
    y = np.asarray(y)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

probabilities = np.array([0.01, 0.10, 0.50, 0.90, 0.99])

pd.DataFrame({
    'p': probabilities,
    'BCE when y=1': binary_cross_entropy_from_probability(probabilities, 1),
    'BCE when y=0': binary_cross_entropy_from_probability(probabilities, 0),
})


In [ ]:
p_grid = np.linspace(0.001, 0.999, 500)

fig, ax = plt.subplots()
ax.plot(p_grid, binary_cross_entropy_from_probability(p_grid, 1), label='y=1')
ax.plot(p_grid, binary_cross_entropy_from_probability(p_grid, 0), label='y=0')
ax.set_xlabel('Predicted probability p')
ax.set_ylabel('Binary cross-entropy')
ax.set_title('Confident wrong predictions receive large penalties')
ax.legend()
plt.show()


## 4. Logits versus probabilities

A logit is a raw score before sigmoid. Frameworks often prefer logits-aware losses for numerical stability. PyTorch's `BCEWithLogitsLoss` is equivalent to applying sigmoid and then BCE, but computed stably.


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce_from_logits_stable(logits, y):
    logits = np.asarray(logits)
    y = np.asarray(y)
    # Stable form: max(z, 0) - z*y + log(1 + exp(-abs(z)))
    return np.maximum(logits, 0) - logits * y + np.log1p(np.exp(-np.abs(logits)))

logits = np.array([-8.0, -2.0, 0.0, 2.0, 8.0])
y = np.array([0, 0, 1, 1, 1])

pd.DataFrame({
    'logit': logits,
    'sigmoid(logit)': sigmoid(logits),
    'target': y,
    'BCE(sigmoid(logit), y)': binary_cross_entropy_from_probability(sigmoid(logits), y),
    'stable BCE from logits': bce_from_logits_stable(logits, y),
})


## 5. Multi-class softmax and categorical cross-entropy

For mutually exclusive classes, softmax turns `K` logits into one probability distribution. Categorical cross-entropy is the negative log-probability of the correct class.


In [ ]:
def softmax(logits, axis=1):
    logits = np.asarray(logits)
    shifted = logits - np.max(logits, axis=axis, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=axis, keepdims=True)

def cross_entropy_from_probabilities(probs, class_indices, eps=1e-12):
    probs = np.clip(np.asarray(probs), eps, 1.0)
    class_indices = np.asarray(class_indices)
    return -np.log(probs[np.arange(len(class_indices)), class_indices])

def cross_entropy_from_logits(logits, class_indices):
    probs = softmax(logits, axis=1)
    return cross_entropy_from_probabilities(probs, class_indices)

logits_mc = np.array([
    [3.0, 1.0, -1.0],
    [0.2, 2.0, 0.1],
    [2.0, 2.1, 2.2],
])
y_mc = np.array([0, 1, 2])
probs_mc = softmax(logits_mc)

pd.DataFrame(probs_mc, columns=['class 0', 'class 1', 'class 2']).assign(
    target=y_mc,
    loss=cross_entropy_from_logits(logits_mc, y_mc),
)


In [ ]:
print('Each softmax row sums to:', probs_mc.sum(axis=1))
print('Mean multi-class cross-entropy:', cross_entropy_from_logits(logits_mc, y_mc).mean())


## 6. Why independent sigmoids are wrong for standard multi-class classification

If classes are mutually exclusive, class probabilities should compete and sum to one. Independent sigmoid outputs do not enforce that.


In [ ]:
independent_logits = np.array([[3.0, 2.5, 2.0]])
independent_sigmoids = sigmoid(independent_logits)
softmax_probs = softmax(independent_logits)

pd.DataFrame({
    'class': ['class 0', 'class 1', 'class 2'],
    'independent sigmoid output': independent_sigmoids.ravel(),
    'softmax probability': softmax_probs.ravel(),
})


In [ ]:
print('Sum of independent sigmoid outputs:', independent_sigmoids.sum())
print('Sum of softmax probabilities:', softmax_probs.sum())


## 7. Common combinations and diagnostics

Use this table as a quick debugging checklist.


In [ ]:
diagnostics = pd.DataFrame([
    {
        'task': 'regression',
        'recommended output': 'linear',
        'recommended loss': 'MSE',
        'common mistake': 'sigmoid output restricts predictions to (0, 1)',
    },
    {
        'task': 'binary classification',
        'recommended output': 'one logit or sigmoid probability',
        'recommended loss': 'BCEWithLogits or BCE',
        'common mistake': 'ReLU output cannot represent calibrated probabilities',
    },
    {
        'task': 'multi-class classification',
        'recommended output': 'K logits or softmax probabilities',
        'recommended loss': 'CrossEntropy or categorical CE',
        'common mistake': 'independent sigmoids for mutually exclusive classes',
    },
])
diagnostics


## 8. Numerical stability demo

For extreme logits, direct sigmoid then log can underflow or produce clipped values. The stable logits formula avoids that problem.


In [ ]:
extreme_logits = np.array([-1000.0, -100.0, 0.0, 100.0, 1000.0])
targets = np.array([0, 0, 1, 1, 1])

# Probability route with clipping is finite, but it has already lost exact probability information.
prob_route = binary_cross_entropy_from_probability(sigmoid(np.clip(extreme_logits, -700, 700)), targets)
stable_route = bce_from_logits_stable(extreme_logits, targets)

pd.DataFrame({
    'logit': extreme_logits,
    'target': targets,
    'probability_route_with_clipping': prob_route,
    'stable_logits_route': stable_route,
})


## Exercises

1. For a 4-class problem, create logits for two examples and compute softmax probabilities and cross-entropy by hand with the helper functions.
2. Change a correct-class probability from 0.9 to 0.1. How much does cross-entropy increase?
3. Explain why PyTorch's `CrossEntropyLoss` should receive logits rather than already-softmaxed probabilities.


## Takeaways

- Loss functions must match the task and output representation.
- Binary cross-entropy heavily penalizes confident wrong predictions.
- Softmax is appropriate for mutually exclusive multi-class probabilities.
- Logits-aware losses are preferred in implementation because they are numerically stable.
